# Phase 2A — Exploratory Data Analysis & Operational Delay Hotspots

**Project:** Airline Delay Prediction & Operations Analytics  
**Scope:** Pre-departure feature relationships, temporal bottlenecks, carrier/airport variations, and operational insights.  
**Dataset:** Validated ML-ready features (`data/processed/flights_features.parquet`) and operational records (`data/processed/flights_cleaned_operational.parquet`).

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Configure backend for non-blocking execution in automated pipelines
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')

# Set workspace root
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print('Environment and libraries initialized successfully.')

## 1. Dataset Loading & Executive Overview

We examine the distribution between valid completed flights, cancelled flights, diverted flights, and the binary delay target.

In [2]:
features_path = project_root / 'data' / 'processed' / 'flights_features.parquet'
ops_path = project_root / 'data' / 'processed' / 'flights_cleaned_operational.parquet'

features_df = pd.read_parquet(features_path)
ops_df = pd.read_parquet(ops_path) if ops_path.exists() else None

total_valid = len(features_df)
delayed_count = int((features_df['delay_target'] == 1).sum())
ontime_count = int((features_df['delay_target'] == 0).sum())
delay_rate = (delayed_count / total_valid) * 100

print(f'Total Valid Completed Flights : {total_valid:,}')
print(f'Delayed Flights (>=15 min)    : {delayed_count:,} ({delay_rate:.2f}%)')
print(f'On-Time Flights (<15 min)     : {ontime_count:,} ({100 - delay_rate:.2f}%)')
if ops_df is not None:
    n_cancelled = int(ops_df.get('_is_cancelled', pd.Series(0)).sum())
    n_diverted = int(ops_df.get('_is_diverted', pd.Series(0)).sum())
    print(f'Cancelled Flights Segregated  : {n_cancelled:,}')
    print(f'Diverted Flights Segregated   : {n_diverted:,}')

## 2. Temporal Delay Analysis

Aviation networks exhibit substantial intra-day delay propagation as turnaround buffers absorb early upstream delays.

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Delay by Hour
hourly = features_df.groupby('departure_hour')['delay_target'].agg(['mean', 'count']).reset_index()
hourly['delay_pct'] = hourly['mean'] * 100
sns.barplot(data=hourly, x='departure_hour', y='delay_pct', ax=ax1, color='#2c3e50')
ax1.set_title('Arrival Delay Rate by Scheduled Departure Hour', fontweight='bold')
ax1.set_xlabel('Scheduled Departure Hour')
ax1.set_ylabel('Delay Rate (%)')
ax1.axhline(delay_rate, color='red', linestyle='--', label=f'Avg ({delay_rate:.1f}%)')
ax1.legend()

# Delay by Time of Day
tod_order = ['morning', 'afternoon', 'evening', 'overnight']
tod = features_df.groupby('time_of_day')['delay_target'].agg(['mean', 'count']).reindex(tod_order).dropna().reset_index()
tod['delay_pct'] = tod['mean'] * 100
sns.barplot(data=tod, x='time_of_day', y='delay_pct', ax=ax2, hue='time_of_day', legend=False, palette='Blues_d')
ax2.set_title('Delay Rate by Operational Time Block', fontweight='bold')
ax2.set_xlabel('Time of Day')
ax2.set_ylabel('Delay Rate (%)')
for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()

## 3. Airline Performance Analysis

> **Important Caveat:** Carrier comparisons are strictly correlational. Observed delay rate variations reflect route network structures, hub hub-congestion, weather exposure, and fleet utilization — not necessarily intrinsic airline operational quality.

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

carrier_stats = features_df.groupby('airline')['delay_target'].agg(['count', 'mean']).reset_index()
carrier_stats['delay_pct'] = carrier_stats['mean'] * 100
carrier_stats = carrier_stats.sort_values('count', ascending=False)

sns.barplot(data=carrier_stats, x='airline', y='count', ax=ax1, color='#2980b9')
ax1.set_title('Flight Volume by Carrier', fontweight='bold')
ax1.set_xlabel('Airline Code')
ax1.set_ylabel('Completed Flights')

carrier_rate_sorted = carrier_stats.sort_values('delay_pct', ascending=False)
sns.barplot(data=carrier_rate_sorted, x='airline', y='delay_pct', ax=ax2, hue='airline', legend=False, palette='Reds_d')
ax2.set_title('Carrier Delay Rate (Correlational)', fontweight='bold')
ax2.set_xlabel('Airline Code')
ax2.set_ylabel('Delay Rate (%)')
ax2.axhline(delay_rate, color='blue', linestyle='--', label=f'Avg ({delay_rate:.1f}%)')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Airport Congestion & Hub Hotspots

We evaluate departure and arrival delay tendencies across major hub airports with minimum sample support.

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

min_flights = 10
origin_df = features_df.groupby('origin_airport')['delay_target'].agg(['count', 'mean']).reset_index()
origin_df['delay_pct'] = origin_df['mean'] * 100
top_origins = origin_df[origin_df['count'] >= min_flights].sort_values('delay_pct', ascending=False).head(10)

sns.barplot(data=top_origins, x='origin_airport', y='delay_pct', ax=ax1, color='#8e44ad')
ax1.set_title(f'Origin Airport Delay Rate (Min {min_flights} Flights)', fontweight='bold')
ax1.set_xlabel('Origin IATA')
ax1.set_ylabel('Delay Rate (%)')

dest_df = features_df.groupby('dest_airport')['delay_target'].agg(['count', 'mean']).reset_index()
dest_df['delay_pct'] = dest_df['mean'] * 100
top_dests = dest_df[dest_df['count'] >= min_flights].sort_values('delay_pct', ascending=False).head(10)

sns.barplot(data=top_dests, x='dest_airport', y='delay_pct', ax=ax2, color='#16a085')
ax2.set_title(f'Destination Airport Delay Rate (Min {min_flights} Flights)', fontweight='bold')
ax2.set_xlabel('Destination IATA')
ax2.set_ylabel('Delay Rate (%)')

plt.tight_layout()
plt.show()

## 5. Flight Distance & Route Haul Analysis

Longer flights provide more airborne en-route opportunity to make up for minor departure delays, whereas short-haul flights have tighter turnarounds.

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=features_df, x='delay_target', y='distance', ax=ax1, hue='delay_target', legend=False, palette=['#2ecc71', '#e74c3c'])
ax1.set_xticks([0, 1])
ax1.set_xticklabels(['On-Time (0)', 'Delayed (1)'])
ax1.set_title('Flight Distance Distribution by Target', fontweight='bold')
ax1.set_ylabel('Statute Miles')

haul = features_df.groupby('haul_category')['delay_target'].agg(['count', 'mean']).reindex(['short_haul', 'medium_haul', 'long_haul']).dropna().reset_index()
haul['delay_pct'] = haul['mean'] * 100
sns.barplot(data=haul, x='haul_category', y='delay_pct', ax=ax2, hue='haul_category', legend=False, palette='Purples_d')
ax2.set_title('Delay Rate by Distance Haul Category', fontweight='bold')
ax2.set_xlabel('Haul Category')
ax2.set_ylabel('Delay Rate (%)')
for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()

## 6. Correlation Matrix & Feature Interdependencies

We assess linear and rank correlations between timing, distance, historical features, and delay outcome.

In [7]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_cols = [
    'delay_target',
    'departure_hour',
    'departure_hour_sin',
    'departure_hour_cos',
    'day_of_week',
    'day_of_week_sin',
    'day_of_week_cos',
    'distance',
    'historical_origin_delay_rate',
    'historical_airline_delay_rate',
    'historical_destination_delay_rate',
]
corr_matrix = features_df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.3, vmax=0.3, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Matrix with Delay Target', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Weather Foundation & Sample Limitations Summary

- **Weather Integration Status:** `WEATHER STATUS: FOUNDATION READY — REAL DATA NOT PROVIDED`
  Real external weather data is not currently present in `data/external/`. Per specification, no synthetic official weather records were fabricated into the production dataset.
- **Historical Features Limitation:** The development sample contains 481 valid flights across 10 days. Trailing historical delay rates function correctly with strict anti-leakage boundaries ($t_{obs} < T_{pred}$), but sample-derived rates should not be treated as population-level estimates.
- **Ready for Phase 2B:** All 38 features are validated, non-leaky, and ready for model training.